# Processamento de um lote PNCP com Docling

Use este notebook para um ZIP criado por `pipeline_lotes_documentos.py`. O estado de extração é isolado por período e lote.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import subprocess
import sys
from pathlib import Path

RAIZ_DRIVE = Path('/content/drive/MyDrive/pncp_rag')
PROJETO_DRIVE = RAIZ_DRIVE / 'projeto'
if not PROJETO_DRIVE.is_dir():
    raise FileNotFoundError(f'Projeto nao encontrado: {PROJETO_DRIVE}')

sys.path.insert(0, str(PROJETO_DRIVE))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJETO_DRIVE / 'requirements-colab.txt')], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU indisponivel. Selecione T4 GPU no menu Runtime.')
print('GPU em uso:', torch.cuda.get_device_name(0))


GPU em uso: Tesla T4


In [ ]:
# ---------------- CONFIGURACAO DO LOTE ----------------
ANO = 2026
MES = 8
NOME_LOTE = 'lote_002'
LIMITE_DOCUMENTOS = 50    # quantidade maxima desta execucao dentro do ZIP
TAMANHO_MICROLOTE = 10
PERFIL = 'qualidade'
TIMEOUT_DOCUMENTO = 180   # em segundos; planta/PDF lento vira timeout
REPROCESSAR_ERROS = False
REPROCESSAR_TIMEOUTS = False  # so True para tentar timeouts conscientemente

ARQUIVO_ZIP_DRIVE = RAIZ_DRIVE / 'lotes' / f'{ANO}_{MES:02d}' / f'documentos_{NOME_LOTE}.zip'
RAIZ_DOCUMENTOS_COLAB = Path('/content/pncp_documentos')
if not ARQUIVO_ZIP_DRIVE.is_file():
    raise FileNotFoundError(f'ZIP do lote nao encontrado: {ARQUIVO_ZIP_DRIVE}')


In [ ]:
import shutil
import zipfile

ZIP_LOCAL = Path('/content/documentos.zip')
shutil.copy2(ARQUIVO_ZIP_DRIVE, ZIP_LOCAL)
shutil.rmtree(RAIZ_DOCUMENTOS_COLAB, ignore_errors=True)
RAIZ_DOCUMENTOS_COLAB.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_LOCAL) as arquivo_zip:
    arquivo_zip.extractall(RAIZ_DOCUMENTOS_COLAB)
print('Documentos prontos em:', RAIZ_DOCUMENTOS_COLAB)


Documentos prontos em: /content/pncp_documentos


In [ ]:
from src.base_estruturada.pipelines.pipeline_documentos import PeriodoColeta, caminhos_do_periodo
from src.base_estruturada.pipelines.pipeline_lotes_documentos import (
    caminhos_do_lote, carregar_lote, gerar_fila_docling_do_lote,
)
from src.base_estruturada.pipelines.pipeline_docling_lotes import executar_docling_por_lotes

periodo = PeriodoColeta(ano=ANO, mes=MES)
caminhos_periodo = caminhos_do_periodo(periodo)
caminhos_lote = caminhos_do_lote(periodo, NOME_LOTE)
lote = carregar_lote(PROJETO_DRIVE / caminhos_lote['lote'])
fila = gerar_fila_docling_do_lote(lote, PROJETO_DRIVE / caminhos_lote['fila_docling'])
print('PDFs na fila do lote:', fila['quantidadeDocumentos'])


PDFs na fila do lote: 654


In [ ]:
import importlib
import src.base_estruturada.pipelines.pipeline_docling_lotes as pipeline

importlib.reload(pipeline)
executar_docling_por_lotes = pipeline.executar_docling_por_lotes

In [ ]:
resumo = executar_docling_por_lotes(
    fila=fila,
    caminho_estado=PROJETO_DRIVE / caminhos_lote['estado_docling'],
    caminho_relatorio=PROJETO_DRIVE / caminhos_lote['relatorio_docling'],
    raiz_documentos=RAIZ_DOCUMENTOS_COLAB,
    pasta_saida=PROJETO_DRIVE / caminhos_periodo['pasta_textos_docling'],
    raiz_projeto_portavel=PROJETO_DRIVE,
    tamanho_microlote=TAMANHO_MICROLOTE,
    limite_documentos=LIMITE_DOCUMENTOS,
    perfil=PERFIL,
    timeout_documento=TIMEOUT_DOCUMENTO,
    reprocessar_erros=REPROCESSAR_ERROS,
    reprocessar_timeouts=REPROCESSAR_TIMEOUTS,
)
print('Resumo final:', resumo)


[1/50] INICIANDO | microlote=1 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010373-2026/TR795380_000090_2026_assinado_assinado.pdf


[INFO] 2026-10-06 21:23:29,574 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-10-06 21:23:29,575 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-10-06 21:23:29,631 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-10-06 21:23:29,632 [RapidOCR] main.py:50: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-10-06 21:23:29,925 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-10-06 21:23:29,927 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-10-06 21:23:29,934 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-10-06 21:23:29,936 [RapidOCR] main.py:50: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-10-06 21

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[1/50] CONCLUIDO | 10.5s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010373-2026/TR795380_000090_2026_assinado_assinado.pdf
[2/50] INICIANDO | microlote=1 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010374-2026/04 - TR_COZ(1).assinado.pdf
[2/50] CONCLUIDO | 11.1s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010374-2026/04 - TR_COZ(1).assinado.pdf
[3/50] INICIANDO | microlote=1 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010380-2026/04-_TR_791600-000059-2026%281%29(1).assinado.pdf
[3/50] CONCLUIDO | 6.4s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010380-2026/04-_TR_791600-000059-2026%281%29(1).assinado.pdf
[4/50] INICIANDO | microlote=1 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010382-2026/07---TR---BAUER.pdf-print.pdf
[4/50] CONCLUIDO | 29.2s | /content/pncp_documentos/original/contratacoes/2026/08/00394502

[WARNING] 2026-10-06 21:24:34,346 [RapidOCR] main.py:132: The text detection result is empty


...

[35/50] CONCLUIDO | 27.1s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010476-2026/19-2026.pdf
[36/50] INICIANDO | microlote=4 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010477-2026/03-Termo-de-Referencia-IMPRESSORA_2026_assinado_assinado_assinado_assinado_assinado.pdf


[36/50] CONCLUIDO | 9.1s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010477-2026/03-Termo-de-Referencia-IMPRESSORA_2026_assinado_assinado_assinado_assinado_assinado.pdf
[37/50] INICIANDO | microlote=4 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010479-2026/7_-_TR_744030-000170-2026_assinado.pdf
[37/50] CONCLUIDO | 12.7s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010479-2026/7_-_TR_744030-000170-2026_assinado.pdf
[38/50] INICIANDO | microlote=4 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010488-2026/56-2026.pdf
[38/50] CONCLUIDO | 6.9s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010488-2026/56-2026.pdf
[39/50] INICIANDO | microlote=4 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010493-2026/48-2026.pdf


[WARNING] 2026-10-06 21:31:52,596 [RapidOCR] main.py:132: The text detection result is empty


[39/50] CONCLUIDO | 5.0s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010493-2026/48-2026.pdf
[40/50] INICIANDO | microlote=4 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010495-2026/70-2026.pdf
[40/50] CONCLUIDO | 5.8s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010495-2026/70-2026.pdf
Microlote 4 | documentos=10 | concluidos=40 | erros=0
[41/50] INICIANDO | microlote=5 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010510-2026/70-2026.pdf
[41/50] CONCLUIDO | 8.3s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010510-2026/70-2026.pdf
[42/50] INICIANDO | microlote=5 | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010511-2026/71-2026.pdf
[42/50] CONCLUIDO | 9.2s | /content/pncp_documentos/original/contratacoes/2026/08/00394502000144-1-010511-2026/71-2026.pdf
[43/50] INICIANDO | microlote=5 | /content/pncp_documentos/origi

[48/50] CONCLUIDO | 140.9s | /content/pncp_documentos/expanded/contratacoes/2026/08/00394502000144-1-010518-2026/TERMO DE REFERÊNCIA.pdf
[49/50] INICIANDO | microlote=5 | /content/pncp_documentos/expanded/contratacoes/2026/08/00394502000144-1-010519-2026/056. EDITAL_79-2026.pdf
[49/50] CONCLUIDO | 3.8s | /content/pncp_documentos/expanded/contratacoes/2026/08/00394502000144-1-010519-2026/056. EDITAL_79-2026.pdf
[50/50] INICIANDO | microlote=5 | /content/pncp_documentos/expanded/contratacoes/2026/08/00394502000144-1-010519-2026/057. ANEXO_I_TR_376-2026.pdf
[50/50] CONCLUIDO | 9.3s | /content/pncp_documentos/expanded/contratacoes/2026/08/00394502000144-1-010519-2026/057. ANEXO_I_TR_376-2026.pdf
Microlote 5 | documentos=10 | concluidos=50 | erros=0
Resumo final: {'selecionados': 50, 'concluidos': 50, 'erros': 0, 'arquivosNaoEncontrados': 0, 'microlotes': 5}
